## The State of Tax Justice: Estimate misalignment for 2020

- Author: Mario Cuenda García, based on Alison Schultz, based on Javier Garcia Bernado's work
- Created: 4 November 2024
- Last updated:

**Description**
- This notebook is the third out of three notebooks to estimate the tax losses caused by profit shifting by multinational enterprises (MNEs). The analysis used the misalignment method based on the country-by-country reports (CbCR) published by the OECD.
    - Details on the misalignment method and its background can be found here: https://www.sciencedirect.com/science/article/pii/S0305750X23003455. 
    - The working paper version is here: https://www.econstor.eu/bitstream/10419/286362/1/wp-2023-33.pdf 

- This notebook estimates profit misalignment based on different formulas. It uses the dataset **"data/final/cbcr_main.csv"** (for the estimation with imputed values) or the dataset **"data/final/cbcr_main_noimputation_allsubgroupsonly.csv"** (for the estimation without imputed values). 

**Outline**
1. Define misalignment
2. Calculate misalignment for sample with full information.
3. Calculate misalignment for samples with imputed data and aggregate results. 

**To dos before running this notebook**
- Run the notebooks 1_clean and 2_impute_missings. Note the requirements and instructions given in these notebooks.

**To dos in this notebook**

Change the formula to any required formula in each section and adapt the output name of the csvs. The formulas I have used are like follows, where sales refer to unrelated party revenues and assets to tangible assets excluding cash.

The order of the formula is as follow:

- formula_vars=['n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll", 'stated_capital' 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip']

    - SOTJ: 50% employees, 50% payroll

        - weights=[1/2, 0, 0, 1/2, 0, 0, 0, 0]

    - Canadian formula: 50% employees, 50% sales

        - weights=[1/2, 1/2, 0, 0, 0, 0, 0, 0]

    - CCCTB: 1/6 employees, 1/6 payroll, 1/3 sales, 1/3 assets

        - weights=[1/6, 1/3, 1/3, 1/6, 0, 0, 0, 0]
        
    - Double-weighted sales: 1/4 employees, 1/2 sales, 1/4 assets

        - weights=[1/4, 1/2, 1/4, 0, 0, 0, 0, 0]

    - Three-factor: 1/3 employees, 1/3 sales, 1/3 assets

        - weights=[1/3, 1/3, 1/3, 0, 0, 0, 0, 0]

- Adjust the input path in 5.1 to the bootstrapped sample you use

## 0. Load packages

In [1]:
# Packages
import pandas as pd
import numpy as np
import tjn_tools
from config import *

# Show columns and select data format
pd.set_option('display.max_columns', None)
pd.options.display.float_format = '{:,.0f}'.format

[TJN TOOLS: Data processing] Module loaded.
[TJN TOOLS: Other functions] Module loaded.
[TJN TOOLS: Paths] Module loaded. Sharepoint FOUND at /Users/mariocuendagarcia/Library/CloudStorage/OneDrive-SharedLibraries-TaxJusticeNetworkLtd


## Step 1. Generate the template datasets

### Step 1.1. Generate the dataset with Unique ISO parents

In [2]:
# Open the original dataset
iso_parents = pd.read_csv(f'{data_final}/cbcr_main_no_imputation_allsubgroupsonly.csv')

# Keep just the following columns: iso_parent and year
iso_parents = iso_parents[['iso_parent', 'year']]

# Keep every unique combination of iso_parent and year
iso_parents = iso_parents.drop_duplicates(subset=['iso_parent', 'year'])

# Sort by year, then iso_parent
iso_parents = iso_parents.sort_values(by=['year', 'iso_parent'])

# Filter by year
iso_parents_2016= iso_parents[iso_parents['year'] == 2016]
iso_parents_2017= iso_parents[iso_parents['year'] == 2017]
iso_parents_2018= iso_parents[iso_parents['year'] == 2018]
iso_parents_2019= iso_parents[iso_parents['year'] == 2019]
iso_parents_2020= iso_parents[iso_parents['year'] == 2020]
iso_parents_2021= iso_parents[iso_parents['year'] == 2021]

# OPTIONAL: Print the count of how many unique iso_partner values there are
print(iso_parents_2016['iso_parent'].nunique())
print(iso_parents_2017['iso_parent'].nunique())
print(iso_parents_2018['iso_parent'].nunique())
print(iso_parents_2019['iso_parent'].nunique())
print(iso_parents_2020['iso_parent'].nunique())
print(iso_parents_2021['iso_parent'].nunique())

iso_parents_2020

26
38
46
50
52
52


,iso_parent,year
154,ARG,2020
268,AUS,2020
764,AUT,2020
838,BEL,2020
1007,BGR,2020
1067,BMU,2020
1648,BRA,2020
1897,CAN,2020
1988,CHE,2020
2726,CHL,2020


### Step 1.2. Generate the dataset with unique iso_partners

In [3]:
## Download the relevant dataset
iso_partners = pd.read_csv(f'{data_final}/cbcr_main_no_imputation_allsubgroupsonly.csv')

# Exclude country groups
iso_partners = iso_partners[~iso_partners['iso_partner'].isin(non_countries)]

# Keep just the following columns: iso_partner and year
iso_partners = iso_partners[['iso_partner', 'year']]

# Sort by year, then iso_partner
iso_partners = iso_partners.sort_values(by=['year', 'iso_partner'])

# Keep every unique combination of iso_partner and year
iso_partners = iso_partners.drop_duplicates(subset=['iso_partner', 'year'])

# Filter by year
iso_partners_2016= iso_partners[iso_partners['year'] == 2016]
iso_partners_2017= iso_partners[iso_partners['year'] == 2017]
iso_partners_2018= iso_partners[iso_partners['year'] == 2018]
iso_partners_2019= iso_partners[iso_partners['year'] == 2019]
iso_partners_2020= iso_partners[iso_partners['year'] == 2020]
iso_partners_2021= iso_partners[iso_partners['year'] == 2021]


# Optional: Print the count of how many unique iso_partner values there are 
print(iso_partners_2016['iso_partner'].nunique())
print(iso_partners_2017['iso_partner'].nunique())
print(iso_partners_2018['iso_partner'].nunique())
print(iso_partners_2019['iso_partner'].nunique())
print(iso_partners_2020['iso_partner'].nunique())
print(iso_partners_2021['iso_partner'].nunique())

183
215
213
210
212
211


### Step 1.3. Generate the template dataset

In [4]:
# Perform a cross join to merge all values of iso_partners_ with each value of iso_parents
iso_combinations_2016 = iso_parents_2016.assign(key=1).merge(iso_partners_2016.assign(key=1), on='key').drop('key', axis=1)
iso_combinations_2017 = iso_parents_2017.assign(key=1).merge(iso_partners_2017.assign(key=1), on='key').drop('key', axis=1)
iso_combinations_2018 = iso_parents_2018.assign(key=1).merge(iso_partners_2018.assign(key=1), on='key').drop('key', axis=1)
iso_combinations_2019 = iso_parents_2019.assign(key=1).merge(iso_partners_2019.assign(key=1), on='key').drop('key', axis=1)
iso_combinations_2020 = iso_parents_2020.assign(key=1).merge(iso_partners_2020.assign(key=1), on='key').drop('key', axis=1)
iso_combinations_2021 = iso_parents_2021.assign(key=1).merge(iso_partners_2021.assign(key=1), on='key').drop('key', axis=1)

# Concatenate all the years
template_dataset = pd.concat([iso_combinations_2016, iso_combinations_2017, iso_combinations_2018, iso_combinations_2019, iso_combinations_2020, iso_combinations_2021])

# Drop year_y
template_dataset = template_dataset.drop(columns=['year_y'])
# Rename year_x to year
template_dataset = template_dataset.rename(columns={'year_x': 'year'})
# Order columns by iso_parent then iso_partner then year
template_dataset = template_dataset[['iso_parent', 'iso_partner', 'year']]
# Generate new column called cbcr_estimates
template_dataset['cbcr_estimates'] = np.nan

template_dataset

,iso_parent,iso_partner,year,cbcr_estimates
0,AUS,ABW,2016,NaN
1,AUS,AFG,2016,NaN
2,AUS,AGO,2016,NaN
3,AUS,ALB,2016,NaN
4,AUS,AND,2016,NaN
...,...,...,...,...
10967,ZAF,XKV,2021,NaN
10968,ZAF,YEM,2021,NaN
10969,ZAF,ZAF,2021,NaN
10970,ZAF,ZMB,2021,NaN


## Step 2. Define the misalignment formula

In [5]:
def calculate_misalignment(cbcr_data,
                           formula_vars=['n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll",
                                         'stated_capital', 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip'],
                           weights=[.5, 0, 0, .5, 0, 0, 0, 0],
                           profit_var='profit_loss_before_income_tax_corrected',
                           etr_max=0.15): 

    # Create variable with positive profits only for calculating shares
    cbcr_data['profit_var_pos'] = cbcr_data[profit_var]
    cbcr_data.loc[cbcr_data[profit_var] < 0, 'profit_var_pos'] = 0
    cbcr_data['share_profit'] = cbcr_data['profit_var_pos'] / cbcr_data.groupby('iso_parent')['profit_var_pos'].transform('sum')

    # Calculate weighted shares of economic activity
    actual_weights = []
    actual_variables = []
    for i, var in enumerate(formula_vars):
        if var is not None and weights[i] > 0:
            actual_variables.append(f"share_{var}")
            actual_weights.append(weights[i])
            cbcr_data.loc[cbcr_data[var] < 0, var] = 0  # Set economic activity measure to zero if negative
            cbcr_data[f"share_{var}"] = cbcr_data[var] / cbcr_data.groupby('iso_parent')[var].transform('sum')

    # Calculate the share of economic activity
    cbcr_data["share_economy_partner_of_parent"] = (cbcr_data.loc[:, actual_variables] * actual_weights).sum(1, min_count=len(actual_weights))
    # Set economic activity to 1% for those jurisdictions without economic activity but with reported profits
    cbcr_data.loc[(cbcr_data["share_economy_partner_of_parent"] == 0) & (cbcr_data[profit_var] > 0), "share_economy_partner_of_parent"] = 0.01

    # Normalize the economic activity shares to sum to 1
    cbcr_data["share_economy_partner_of_parent"] = cbcr_data["share_economy_partner_of_parent"] / cbcr_data.groupby('iso_parent')["share_economy_partner_of_parent"].transform('sum')

    # Calculate theoretical profit and misaligned profit
    cbcr_data["theoretical_profit"] = cbcr_data["share_economy_partner_of_parent"] * cbcr_data.groupby('iso_parent')[profit_var].transform('sum')
    cbcr_data["misaligned_profit"] = cbcr_data[profit_var] - cbcr_data["theoretical_profit"]

    # Set positive misaligned profits to 0 if ETR exceeds the threshold (etr_max)
    cbcr_data.loc[((cbcr_data["misaligned_profit"] > 0) & (cbcr_data["etr_average_corrected"] > etr_max)), "misaligned_profit"] = 0

    # Adjust misalignment per 'iso_parent'
    def adjust_misalignment(group):
        total_negative_misalignment = group.loc[group["misaligned_profit"] < 0, "misaligned_profit"].sum()
        total_positive_misalignment = group.loc[group["misaligned_profit"] > 0, "misaligned_profit"].sum()
        
        # Adjust negative misalignments to balance positive misalignments within each 'iso_parent'
        if total_negative_misalignment != 0:
            factor = - total_positive_misalignment / total_negative_misalignment
            group.loc[group["misaligned_profit"] < 0, "misaligned_profit"] *= factor
        
        return group

    # Apply the adjustment by grouping by 'iso_parent'
    cbcr_data = cbcr_data.groupby('iso_parent', group_keys=False).apply(lambda x: adjust_misalignment(x)).reset_index(drop=True)

    return cbcr_data

## Step 3. Calculate misalignment for sample with full information

### Step 3.1 Import data
- Import data without imputed values. This is only the data from the sample of reporting countries that actually is reported on a country basis, i.e. excluding aggregated country groups and data from reporting countries that do not report on a country by country basis, but just by continents.

In [6]:
cbcr_sample = pd.read_csv(f'{data_final}/cbcr_main_no_imputation_allsubgroupsonly.csv')

# Exclude country groups
cbcr_sample = cbcr_sample[~cbcr_sample['iso_partner'].isin(non_countries)]

### Step 3.2 Exclude countries that do not report truly country-by-country


- In the 2024 data, the following reporting countries do not report country-by-country. We exclude those from the "clean" analysis where we only use values that are actually in the data.
    - Austria: Only continents in all years
    - Czechia: Only Czechia versus rest of the world from 2019 to 2021
    - Finland: Only Finland and rest of the world between 2016 and 2018 and Finland and continents between 2019 and 2021
    - Greece: Only Greece and continents between 2017 and 2019
    - Hungary: Only Hungary versus rest of the world between 2018 and 2021
    - Isle of Man: Only continents between 2017 and 2020
    - Ireland: Only Ireland versus rest of the world in all years
    - Korea: Only Korea and rest of the world betweem 2016 and 2018 and Korea and continents between 2019 and 2021
    - Macau: Only Macau versus rest of the world between 2019 and 2021
    - Mauritius: Only Mauritius and continents between 2019 and 2021
    - Morocco: Only Morocco and continents in 2021
    - Netherlands: Only Netherlands versus rest of the world between 2016 and 2017
    - Norway: Only Norway and continents 2016 and 2017
    - New Zealand: Only New Zealand versus rest of the world between 2018 and 2021
    - Poland: Only Poland and continents 2019 to 2021
    - Sweden: Only Sweden and continents in all years
    - United Kingdom: Only UK and continents between 2017 and 2021

In [7]:
# Define the conditions for exclusion
exclusion_conditions = [
    ('AUT', 2016, 2021),                # Austria: all years
    ('CZE', 2019, 2021),                # Czechia: from 2019 to 2021
    ('FIN', 2016, 2021),                # Finland: all years
    ('GRC', 2017, 2019),                # Greece: between 2017 and 2019
    ('HUN', 2018, 2021),                # Hungary: between 2018 and 2021
    ('IMN', 2017, 2020),                # Isle of Man: between 2017 and 2020
    ('IRL', 2016, 2021),                # Ireland: all years
    ('KOR', 2016, 2021),                # Korea: all years
    ('MAC', 2019, 2021),                # Macau: between 2019 and 2021
    ('MUS', 2019, 2021),                # Mauritius: between 2019 and 2021
    ('MAR', 2021, 2021),                # Morocco: 2021
    ('NLD', 2016, 2017),                # Netherlands: between 2016 and 2017
    ('NOR', 2016, 2017),                # Norway: 2016 and 2017
    ('NZL', 2018, 2021),                # New Zealand: between 2018 and 2021
    ('POL', 2019, 2021),                # Poland: 2019 to 2021
    ('SWE', 2016, 2021),                # Sweden: all years
    ('GBR', 2017, 2021)                 # United Kingdom: between 2017 and 2021
]

# Iterate through the exclusion conditions
for iso_parent, start_year, end_year in exclusion_conditions:
    cbcr_sample = cbcr_sample[~((cbcr_sample['iso_parent'] == iso_parent) & 
                                     (cbcr_sample['year'].between(start_year, end_year)))]

### Step 3.3 Calculate misalignment for sample countries with full information

The order of the formula is as follow:

- formula_vars=['n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll", 'stated_capital' 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip']

    - SOTJ: 50% employees, 50% payroll

        - weights=[1/2, 0, 0, 1/2, 0, 0, 0, 0]

    - Canadian formula: 50% employees, 50% sales

        - weights=[1/2, 1/2, 0, 0, 0, 0, 0, 0]

    - CCCTB: 1/6 employees, 1/6 payroll, 1/3 sales, 1/3 assets

        - weights=[1/6, 1/3, 1/3, 1/6, 0, 0, 0, 0]
        
    - Double-weighted sales: 1/4 employees, 1/2 sales, 1/4 assets

        - weights=[1/4, 1/2, 1/4, 0, 0, 0, 0, 0]

    - Three-factor: 1/3 employees, 1/3 sales, 1/3 assets

        - weights=[1/3, 1/3, 1/3, 0, 0, 0, 0, 0]


The end of this cell shows, among other things, the misaligned profits and the theoretical profits, as well as the CBCR variables.

**More importantly, if for whatever reason we wanted to just run this cell without the "bad reporters", we could just modify the cell to run like the cell from Step 5.2. in order to obtain a dataset of profit shifting without the "bad reporters". In a way, this cell is not really necessary for the rest of the notebook. It just shows an intermediary steps if we want to calculate the misalignment for the "good reporters" only.**

In [8]:
misalignment_2020 = cbcr_sample[cbcr_sample['year'] == 2020].copy()
misalignment_2020 = calculate_misalignment(misalignment_2020, etr_max=0.15, weights=[1/2, 0, 0, 1/2, 0, 0, 0, 0])

# Keep only the first occurrence of these unique variables for each 'iso_partner'
unique_columns = misalignment_2020.drop_duplicates(subset=['iso_partner'])[['iso_partner', 'partner_jurisdiction', 
                                                                               'etr_average_corrected', 'cit',
                                                                               'tax_revenue_current_usd', 
                                                                               'gvt_health_expenditure', 'region_tjn', 
                                                                               'ukt', 'oecd', 'oecd_oct', 'nld_oct']]

# Keep iso_parent, iso_partner, year, misaligned_profit, theoretical_profit, profit_loss_before_income_tax_corrected, 'n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll",'stated_capital', 'total_revenues', 'related_party_revenues', and 'holding_or_managing_ip'
misalignment_2020 = misalignment_2020[['iso_parent', 'iso_partner', 'year', 'misaligned_profit', 'theoretical_profit', 'profit_loss_before_income_tax_corrected', 'n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll",'stated_capital', 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip']]

# Show if iso_partner = USA
misalignment_2020[misalignment_2020['iso_partner'] == 'USA']

/var/folders/pm/bp4z4lln39xcwn73chrtp3n00000gn/T/ipykernel_6472/421381151.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent', group_keys=False).apply(lambda x: adjust_misalignment(x)).reset_index(drop=True)


,iso_parent,iso_partner,year,misaligned_profit,theoretical_profit,profit_loss_before_income_tax_corrected,n_employees,unrelated_party_revenues,tangible_assets_except_cash,payroll,stated_capital,total_revenues,related_party_revenues,holding_or_managing_ip
15,ARG,USA,2020,0,"5,508,650","28,958,704",120,"147,400,570","1,922,324,821","6,483,152","1,993,072,796","166,349,934","18,949,363",1
102,AUS,USA,2020,"-1,580,646,524","5,121,522,427","1,389,888,211","83,258","48,052,343,508","38,810,548,367","4,498,119,021","159,990,000,000","65,715,677,829","17,663,334,322",29
131,BEL,USA,2020,0,"2,106,727,235","2,797,200,000","59,600","43,666,000,000","13,784,400,000","3,219,965,573","63,166,000,000","53,844,500,000","10,178,600,000",17
228,BMU,USA,2020,"-1,251,350,565","5,596,417,577","4,250,896,026","90,252","83,031,484,279","41,602,085,998","4,875,978,740","75,291,544,548","104,603,000,000","21,571,590,436",28
265,BRA,USA,2020,"-152,661,451","5,936,000,906","5,502,515,000","122,877","55,703,159,000","20,370,959,000","6,638,585,733","47,721,976,000","69,543,728,000","13,840,568,000",7
278,CAN,USA,2020,"-18,706,763,750","37,894,022,063","24,300,993,000","642,530","388,554,000,000","446,834,000,000","34,713,497,978","1,035,370,000,000","484,333,000,000","95,778,817,000",NaN
412,CHE,USA,2020,0,"11,320,121,954","12,269,427,741","336,749","250,450,000,000","76,190,112,625","18,193,291,723","374,378,000,000","311,570,000,000","61,119,202,330",81
431,CHL,USA,2020,0,"158,864,966","351,693,104","7,709","4,957,403,990","1,705,339,561","416,488,500","2,755,353,122","5,157,368,706","199,964,716",0
557,CHN,USA,2020,"-10,433,978,464","8,023,499,291","-4,918,125,071","73,728","79,786,278,485","56,169,310,571","3,983,248,687","53,415,351,697","109,093,000,000","29,981,989,297",65
695,CYM,USA,2020,"-2,598,480,356","6,196,491,610","-2,057,247,004","65,097","25,890,337,735","20,347,657,095","3,516,947,968","42,842,444,601","39,501,752,491","13,611,414,756",42


## Step 4. Generate the dataset for the "bad reporters"

For the "bad" reporters, we are going to assume that their MNEs behave like the "average" MNE in the countries that report correctly. To do that we need to:

1. First aggregate the variables reported by the CBCR by partner countries. For instance, we see that on aggregate, there are 80m employees reported.
2. Then we look at the share corresponding by partners. For instance, 18m employees are reported in the USA. how many does the USA have. In short, roughly 24% of all employees reported are in the USA.
3. We then assume that 24% of the toal employees reported by the "bad" reporters are assigned to the US. And we repeat with all of them



### Step 4.1. Generate the Total Sums of Variables, and the Total Sums by Partners.

- The first bloc of lines calculates the total of the variables, and generates a new variable (e.g. total_n_employees) in the dataset.
- The second bloc of lines groups by iso_partner and calculates the total sums by partners. (e.g. how many employees are reported by the CBCR countries, say, in the USA)
- The third bloc of lines merges the total sums by partners to the dataset

In [9]:
# Calculate the total sum of profit_loss_before_income_tax_corrected, 'n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll",'stated_capital', 'total_revenues', 'related_party_revenues', and 'holding_or_managing_ip'
total_profit_loss_before_income_tax_corrected = misalignment_2020['profit_loss_before_income_tax_corrected'].sum()
total_n_employees = misalignment_2020['n_employees'].sum()
total_unrelated_party_revenues = misalignment_2020['unrelated_party_revenues'].sum()
total_tangible_assets_except_cash = misalignment_2020['tangible_assets_except_cash'].sum()
total_payroll = misalignment_2020['payroll'].sum()
total_stated_capital = misalignment_2020['stated_capital'].sum()
total_total_revenues = misalignment_2020['total_revenues'].sum()
total_related_party_revenues = misalignment_2020['related_party_revenues'].sum()
total_holding_or_managing_ip = misalignment_2020['holding_or_managing_ip'].sum()

misalignment_2020['total_profit_loss_before_income_tax_corrected'] = total_profit_loss_before_income_tax_corrected
misalignment_2020['total_n_employees'] = total_n_employees
misalignment_2020['total_unrelated_party_revenues'] = total_unrelated_party_revenues
misalignment_2020['total_tangible_assets_except_cash'] = total_tangible_assets_except_cash
misalignment_2020['total_payroll'] = total_payroll
misalignment_2020['total_stated_capital'] = total_stated_capital
misalignment_2020['total_total_revenues'] = total_total_revenues
misalignment_2020['total_related_party_revenues'] = total_related_party_revenues
misalignment_2020['total_holding_or_managing_ip'] = total_holding_or_managing_ip

# Group by iso_partner and calculate the total sum of profit_loss_before_income_tax_corrected, 'n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll",'stated_capital', 'total_revenues', 'related_party_revenues', and 'holding_or_managing_ip'
total_profit_loss_by_partner = misalignment_2020.groupby('iso_partner')['profit_loss_before_income_tax_corrected'].sum().reset_index()
total_profit_loss_by_partner = total_profit_loss_by_partner.rename(columns={'profit_loss_before_income_tax_corrected': 'total_profit_loss_by_partner'})

total_n_employees_by_partner = misalignment_2020.groupby('iso_partner')['n_employees'].sum().reset_index()
total_n_employees_by_partner = total_n_employees_by_partner.rename(columns={'n_employees': 'total_n_employees_by_partner'})

total_unrelated_party_revenues_by_partner = misalignment_2020.groupby('iso_partner')['unrelated_party_revenues'].sum().reset_index()
total_unrelated_party_revenues_by_partner = total_unrelated_party_revenues_by_partner.rename(columns={'unrelated_party_revenues': 'total_unrelated_party_revenues_by_partner'})

total_tangible_assets_except_cash_by_partner = misalignment_2020.groupby('iso_partner')['tangible_assets_except_cash'].sum().reset_index()
total_tangible_assets_except_cash_by_partner = total_tangible_assets_except_cash_by_partner.rename(columns={'tangible_assets_except_cash': 'total_tangible_assets_except_cash_by_partner'})

total_payroll_by_partner = misalignment_2020.groupby('iso_partner')['payroll'].sum().reset_index()
total_payroll_by_partner = total_payroll_by_partner.rename(columns={'payroll': 'total_payroll_by_partner'})

total_stated_capital_by_partner = misalignment_2020.groupby('iso_partner')['stated_capital'].sum().reset_index()
total_stated_capital_by_partner = total_stated_capital_by_partner.rename(columns={'stated_capital': 'total_stated_capital_by_partner'})

total_total_revenues_by_partner = misalignment_2020.groupby('iso_partner')['total_revenues'].sum().reset_index()
total_total_revenues_by_partner = total_total_revenues_by_partner.rename(columns={'total_revenues': 'total_total_revenues_by_partner'})

total_related_party_revenues_by_partner = misalignment_2020.groupby('iso_partner')['related_party_revenues'].sum().reset_index()
total_related_party_revenues_by_partner = total_related_party_revenues_by_partner.rename(columns={'related_party_revenues': 'total_related_party_revenues_by_partner'})

total_holding_or_managing_ip_by_partner = misalignment_2020.groupby('iso_partner')['holding_or_managing_ip'].sum().reset_index()
total_holding_or_managing_ip_by_partner = total_holding_or_managing_ip_by_partner.rename(columns={'holding_or_managing_ip': 'total_holding_or_managing_ip_by_partner'})

# Merge the total profit loss by partner back into the misalignment_2020 dataframe
misalignment_2020 = misalignment_2020.merge(total_profit_loss_by_partner, on='iso_partner', how='left')
misalignment_2020 = misalignment_2020.merge(total_n_employees_by_partner, on='iso_partner', how='left')
misalignment_2020 = misalignment_2020.merge(total_unrelated_party_revenues_by_partner, on='iso_partner', how='left')
misalignment_2020 = misalignment_2020.merge(total_tangible_assets_except_cash_by_partner, on='iso_partner', how='left')
misalignment_2020 = misalignment_2020.merge(total_payroll_by_partner, on='iso_partner', how='left')
misalignment_2020 = misalignment_2020.merge(total_stated_capital_by_partner, on='iso_partner', how='left')
misalignment_2020 = misalignment_2020.merge(total_total_revenues_by_partner, on='iso_partner', how='left')
misalignment_2020 = misalignment_2020.merge(total_related_party_revenues_by_partner, on='iso_partner', how='left')
misalignment_2020 = misalignment_2020.merge(total_holding_or_managing_ip_by_partner, on='iso_partner', how='left')

misalignment_2020[misalignment_2020['iso_partner'] == 'USA']

,iso_parent,iso_partner,year,misaligned_profit,theoretical_profit,profit_loss_before_income_tax_corrected,n_employees,unrelated_party_revenues,tangible_assets_except_cash,payroll,stated_capital,total_revenues,related_party_revenues,holding_or_managing_ip,total_profit_loss_before_income_tax_corrected,total_n_employees,total_unrelated_party_revenues,total_tangible_assets_except_cash,total_payroll,total_stated_capital,total_total_revenues,total_related_party_revenues,total_holding_or_managing_ip,total_profit_loss_by_partner,total_n_employees_by_partner,total_unrelated_party_revenues_by_partner,total_tangible_assets_except_cash_by_partner,total_payroll_by_partner,total_stated_capital_by_partner,total_total_revenues_by_partner,total_related_party_revenues_by_partner,total_holding_or_managing_ip_by_partner
15,ARG,USA,2020,0,"5,508,650","28,958,704",120,"147,400,570","1,922,324,821","6,483,152","1,993,072,796","166,349,934","18,949,363",1,"3,767,972,070,338","141,682,249","51,192,097,093,076","36,847,545,514,002","3,866,168,113,442","94,932,109,105,528","71,330,283,936,715","20,159,725,463,860","19,825","959,487,962,168","28,827,175","14,305,491,574,496","7,891,043,323,455","1,557,424,693,578","20,928,877,633,852","18,589,048,379,595","4,285,242,709,226","1,578"
102,AUS,USA,2020,"-1,580,646,524","5,121,522,427","1,389,888,211","83,258","48,052,343,508","38,810,548,367","4,498,119,021","159,990,000,000","65,715,677,829","17,663,334,322",29,"3,767,972,070,338","141,682,249","51,192,097,093,076","36,847,545,514,002","3,866,168,113,442","94,932,109,105,528","71,330,283,936,715","20,159,725,463,860","19,825","959,487,962,168","28,827,175","14,305,491,574,496","7,891,043,323,455","1,557,424,693,578","20,928,877,633,852","18,589,048,379,595","4,285,242,709,226","1,578"
131,BEL,USA,2020,0,"2,106,727,235","2,797,200,000","59,600","43,666,000,000","13,784,400,000","3,219,965,573","63,166,000,000","53,844,500,000","10,178,600,000",17,"3,767,972,070,338","141,682,249","51,192,097,093,076","36,847,545,514,002","3,866,168,113,442","94,932,109,105,528","71,330,283,936,715","20,159,725,463,860","19,825","959,487,962,168","28,827,175","14,305,491,574,496","7,891,043,323,455","1,557,424,693,578","20,928,877,633,852","18,589,048,379,595","4,285,242,709,226","1,578"
228,BMU,USA,2020,"-1,251,350,565","5,596,417,577","4,250,896,026","90,252","83,031,484,279","41,602,085,998","4,875,978,740","75,291,544,548","104,603,000,000","21,571,590,436",28,"3,767,972,070,338","141,682,249","51,192,097,093,076","36,847,545,514,002","3,866,168,113,442","94,932,109,105,528","71,330,283,936,715","20,159,725,463,860","19,825","959,487,962,168","28,827,175","14,305,491,574,496","7,891,043,323,455","1,557,424,693,578","20,928,877,633,852","18,589,048,379,595","4,285,242,709,226","1,578"
265,BRA,USA,2020,"-152,661,451","5,936,000,906","5,502,515,000","122,877","55,703,159,000","20,370,959,000","6,638,585,733","47,721,976,000","69,543,728,000","13,840,568,000",7,"3,767,972,070,338","141,682,249","51,192,097,093,076","36,847,545,514,002","3,866,168,113,442","94,932,109,105,528","71,330,283,936,715","20,159,725,463,860","19,825","959,487,962,168","28,827,175","14,305,491,574,496","7,891,043,323,455","1,557,424,693,578","20,928,877,633,852","18,589,048,379,595","4,285,242,709,226","1,578"
278,CAN,USA,2020,"-18,706,763,750","37,894,022,063","24,300,993,000","642,530","388,554,000,000","446,834,000,000","34,713,497,978","1,035,370,000,000","484,333,000,000","95,778,817,000",NaN,"3,767,972,070,338","141,682,249","51,192,097,093,076","36,847,545,514,002","3,866,168,113,442","94,932,109,105,528","71,330,283,936,715","20,159,725,463,860","19,825","959,487,962,168","28,827,175","14,305,491,574,496","7,891,043,323,455","1,557,424,693,578","20,928,877,633,852","18,589,048,379,595","4,285,242,709,226","1,578"
412,CHE,USA,2020,0,"11,320,121,954","12,269,427,741","336,749","250,450,000,000","76,190,112,625","18,193,291,723","374,378,000,000","311,570,000,000","61,119,202,330",81,"3,767,972,

### Step 4.2. Calculate the shares for all the variables

In [10]:
# Final Misalignment
final_misalignment_2020 = misalignment_2020

# Calculate the shares for all variables
final_misalignment_2020['share_reported_total_profit_loss_by_partner'] = misalignment_2020['total_profit_loss_by_partner'] / misalignment_2020['total_profit_loss_before_income_tax_corrected']
final_misalignment_2020['share_reported_total_n_employees_by_partner'] = misalignment_2020['total_n_employees_by_partner'] / misalignment_2020['total_n_employees']
final_misalignment_2020['share_reported_total_unrelated_party_revenues_by_partner'] = misalignment_2020['total_unrelated_party_revenues_by_partner'] / misalignment_2020['total_unrelated_party_revenues']
final_misalignment_2020['share_reported_total_tangible_assets_except_cash_by_partner'] = misalignment_2020['total_tangible_assets_except_cash_by_partner'] / misalignment_2020['total_tangible_assets_except_cash']
final_misalignment_2020['share_reported_total_payroll_by_partner'] = misalignment_2020['total_payroll_by_partner'] / misalignment_2020['total_payroll']
final_misalignment_2020['share_reported_total_stated_capital_by_partner'] = misalignment_2020['total_stated_capital_by_partner'] / misalignment_2020['total_stated_capital']
final_misalignment_2020['share_reported_total_total_revenues_by_partner'] = misalignment_2020['total_total_revenues_by_partner'] / misalignment_2020['total_total_revenues']
final_misalignment_2020['share_reported_total_related_party_revenues_by_partner'] = misalignment_2020['total_related_party_revenues_by_partner'] / misalignment_2020['total_related_party_revenues']
final_misalignment_2020['share_reported_total_holding_or_managing_ip_by_partner'] = misalignment_2020['total_holding_or_managing_ip_by_partner'] / misalignment_2020['total_holding_or_managing_ip']

# Give me column 'share reported' with 2 decimals
pd.options.display.float_format = '{:,.2f}'.format

final_misalignment_2020[final_misalignment_2020['iso_partner'] == 'USA']

,iso_parent,iso_partner,year,misaligned_profit,theoretical_profit,profit_loss_before_income_tax_corrected,n_employees,unrelated_party_revenues,tangible_assets_except_cash,payroll,stated_capital,total_revenues,related_party_revenues,holding_or_managing_ip,total_profit_loss_before_income_tax_corrected,total_n_employees,total_unrelated_party_revenues,total_tangible_assets_except_cash,total_payroll,total_stated_capital,total_total_revenues,total_related_party_revenues,total_holding_or_managing_ip,total_profit_loss_by_partner,total_n_employees_by_partner,total_unrelated_party_revenues_by_partner,total_tangible_assets_except_cash_by_partner,total_payroll_by_partner,total_stated_capital_by_partner,total_total_revenues_by_partner,total_related_party_revenues_by_partner,total_holding_or_managing_ip_by_partner,share_reported_total_profit_loss_by_partner,share_reported_total_n_employees_by_partner,share_reported_total_unrelated_party_revenues_by_partner,share_reported_total_tangible_assets_except_cash_by_partner,share_reported_total_payroll_by_partner,share_reported_total_stated_capital_by_partner,share_reported_total_total_revenues_by_partner,share_reported_total_related_party_revenues_by_partner,share_reported_total_holding_or_managing_ip_by_partner
15,ARG,USA,2020,0.00,"5,508,649.77","28,958,704.16",120.00,"147,400,570.40","1,922,324,821.00","6,483,152.16","1,993,072,796.00","166,349,933.80","18,949,363.37",1.00,"3,767,972,070,338.21","141,682,248.52","51,192,097,093,075.98","36,847,545,514,002.43","3,866,168,113,441.89","94,932,109,105,528.05","71,330,283,936,715.33","20,159,725,463,860.24","19,825.00","959,487,962,167.79","28,827,175.21","14,305,491,574,495.95","7,891,043,323,454.89","1,557,424,693,578.42","20,928,877,633,852.16","18,589,048,379,594.77","4,285,242,709,226.02","1,578.00",0.25,0.20,0.28,0.21,0.40,0.22,0.26,0.21,0.08
102,AUS,USA,2020,"-1,580,646,524.37","5,121,522,426.67","1,389,888,211.00","83,258.00","48,052,343,508.00","38,810,548,367.00","4,498,119,021.14","159,990,000,000.00","65,715,677,829.00","17,663,334,322.00",29.00,"3,767,972,070,338.21","141,682,248.52","51,192,097,093,075.98","36,847,545,514,002.43","3,866,168,113,441.89","94,932,109,105,528.05","71,330,283,936,715.33","20,159,725,463,860.24","19,825.00","959,487,962,167.79","28,827,175.21","14,305,491,574,495.95","7,891,043,323,454.89","1,557,424,693,578.42","20,928,877,633,852.16","18,589,048,379,594.77","4,285,242,709,226.02","1,578.00",0.25,0.20,0.28,0.21,0.40,0.22,0.26,0.21,0.08
131,BEL,USA,2020,0.00,"2,106,727,235.19","2,797,200,000.00","59,600.00","43,666,000,000.00","13,784,400,000.00","3,219,965,572.80","63,166,000,000.00","53,844,500,000.00","10,178,600,000.00",17.00,"3,767,972,070,338.21","141,682,248.52","51,192,097,093,075.98","36,847,545,514,002.43","3,866,168,113,441.89","94,932,109,105,528.05","71,330,283,936,715.33","20,159,725,463,860.24","19,825.00","959,487,962,167.79","28,827,175.21","14,305,491,574,495.95","7,891,043,323,454.89","1,557,424,693,578.42","20,928,877,633,852.16","18,589,048,379,594.77","4,285,242,709,226.02","1,578.00",0.25,0.20,0.28,0.21,0.40,0.22,0.26,0.21,0.08
228,BMU,USA,2020,"-1,251,350,565.01","5,596,417,576.94","4,250,896,026.00","90,252.00","83,031,484,279.00","41,602,085,998.00","4,875,978,739.54","75,291,544,548.00","104,603,000,000.00","21,571,590,436.00",28.00,"3,767,972,070,338.21","141,682,248.52","51,192,097,093,075.98","36,847,545,514,002.43","3,866,168,113,441.89","94,932,109,105,528.05","71,330,283,936,715.33","20,159,725,463,860.24","19,825.00","959,487,962,167.79","28,827,175.21","14,305,491,574,495.95","7,891,043,323,454.89","1,557,424,693,578.42","20,928,877,633,852.16","18,589,048,379,594.77","4,285,242,709,226.02","1,578.00",0.25,0.20,0.28,0.21,0.40,0.22,0.26,0.21,0.08
265,BRA,USA,2020,"-152,661,450.88","5,936,000,905.64","5,502,515,000.00","122,877.00","55,703,159,000.00","20,370,959,000.00","6,638,585,733.04","47,721,976,000.00","69,543,728,000.00","13,840,568,000.00",7.00,"3,767,972,070,338.21",

### Step 4.3. Keep only the shares for the partners, and drop duplicates (effectively only keep one value for each iso_partner)c

In [11]:
# Keep iso_partner and share_reported_total_profit_loss_by_partner	share_reported_total_n_employees_by_partner	share_reported_total_unrelated_party_revenues_by_partner	share_reported_total_tangible_assets_except_cash_by_partner	share_reported_total_payroll_by_partner	share_reported_total_stated_capital_by_partner	share_reported_total_total_revenues_by_partner	share_reported_total_related_party_revenues_by_partner	share_reported_total_holding_or_managing_ip_by_partner
shares_reported_2020 = final_misalignment_2020[['iso_partner', 'share_reported_total_profit_loss_by_partner', 'share_reported_total_n_employees_by_partner', 'share_reported_total_unrelated_party_revenues_by_partner', 'share_reported_total_tangible_assets_except_cash_by_partner', 'share_reported_total_payroll_by_partner', 'share_reported_total_stated_capital_by_partner', 'share_reported_total_total_revenues_by_partner', 'share_reported_total_related_party_revenues_by_partner', 'share_reported_total_holding_or_managing_ip_by_partner']]

# Drop duplicates
shares_reported_2020 = shares_reported_2020.drop_duplicates()

shares_reported_2020[shares_reported_2020['iso_partner'] == 'USA']

,iso_partner,share_reported_total_profit_loss_by_partner,share_reported_total_n_employees_by_partner,share_reported_total_unrelated_party_revenues_by_partner,share_reported_total_tangible_assets_except_cash_by_partner,share_reported_total_payroll_by_partner,share_reported_total_stated_capital_by_partner,share_reported_total_total_revenues_by_partner,share_reported_total_related_party_revenues_by_partner,share_reported_total_holding_or_managing_ip_by_partner
15,USA,0.25,0.20,0.28,0.21,0.40,0.22,0.26,0.21,0.08


### Step 4.4. Bring back the excluded countries

- The key here is the third command, where we sum by iso_parent. We basically assume that all countries report for the rest of the world, without caring about continents. **This could be improved and changed**. 
- Once that sum is done, we combine all potential iso_combinations for 2016 (created in Step 1), and we keep all combinations for the "excluded countries".
- Note in the test view, that no matter who is the iso_partner, the number in the variables will be the same because we have aggregated them. The next cells will now create the right shares.

In [12]:
excluded_2020 = pd.read_csv(f'{data_final}/cbcr_main_no_imputation_allsubgroupsonly.csv')

# Keep if year == 2020 and iso_parent == 'AUT', 'FIN', 'IRL0', 'KOR', 'NLD', 'NOR' 'SWE'
excluded_2020 = excluded_2020[excluded_2020['year'] == 2020]
excluded_2020 = excluded_2020[excluded_2020['iso_parent'].isin(['AUT', 'CZE', 'FIN', 'HUN', 'IMN', 'IRL', 'KOR', 'MAC', 'MUS', 'NZL', 'POL', 'SWE', 'GBR'])]

# Sum by iso_parent: 'n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll",'stated_capital', 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip' and 'profit_loss_before_income_tax_corrected'
excluded_2020 = excluded_2020.groupby('iso_parent').agg({'n_employees': 'sum', 'unrelated_party_revenues': 'sum', 'tangible_assets_except_cash': 'sum', 'payroll': 'sum', 'stated_capital': 'sum', 'total_revenues': 'sum', 'related_party_revenues': 'sum', 'holding_or_managing_ip': 'sum', 'profit_loss_before_income_tax_corrected': 'sum'}).reset_index()

# Merge iso_combinations_2020 with excluded_2020. 
excluded_jurisdictions_2020 = pd.merge(iso_combinations_2020, excluded_2020, on='iso_parent', how='left')

# Drop year_y
excluded_jurisdictions_2020 = excluded_jurisdictions_2020.drop(columns=['year_y'])
# Rename year_x to year
excluded_jurisdictions_2020 = excluded_jurisdictions_2020.rename(columns={'year_x': 'year'})

# Keep if iso_parent == 'AUT', 'FIN', 'IRL0', 'KOR', 'NLD', 'NOR' 'SWE'
excluded_jurisdictions_2020 = excluded_jurisdictions_2020[excluded_jurisdictions_2020['iso_parent'].isin(['AUT', 'CZE', 'FIN', 'HUN', 'IMN', 'IRL', 'KOR', 'MAC', 'MUS', 'NZL', 'POL', 'SWE', 'GBR'])]

excluded_jurisdictions_2020[excluded_jurisdictions_2020['iso_partner'] == 'USA']


,iso_parent,year,iso_partner,n_employees,unrelated_party_revenues,tangible_assets_except_cash,payroll,stated_capital,total_revenues,related_party_revenues,holding_or_managing_ip,profit_loss_before_income_tax_corrected
622,AUT,2020,USA,"2,110,422.00","635,656,994,320.30","390,072,595,039.00","21,961,876,537.44","262,331,183,165.00","806,269,144,135.20","170,615,024,109.10",635.00,"34,626,686,395.37"
2742,CZE,2020,USA,"505,983.00","160,491,042,419.00","51,352,333,535.00","1,882,361,260.96","30,830,232,892.00","284,571,256,312.00","124,080,213,894.00",0.00,"5,951,649,725.00"
3590,FIN,2020,USA,"1,011,986.00","453,483,727,687.00","166,982,124,207.00","7,245,292,496.26","629,842,243,437.00","618,489,043,409.00","165,005,293,862.00",220.00,"37,022,941,844.00"
4014,GBR,2020,USA,"13,635,872.00","4,691,030,530,169.00","2,920,722,144,908.00","121,385,259,899.88","12,678,890,253,859.00","6,328,578,247,967.00","1,636,215,717,782.00","2,459.00","364,227,585,609.00"
4650,HUN,2020,USA,"110,735.00","34,980,003,428.00","17,391,156,848.00","731,483,928.12","14,268,893,074.00","47,856,701,945.00","12,876,698,518.00",0.00,"1,914,948,835.00"
5074,IMN,2020,USA,"49,804.00","14,952,480,316.21","11,395,495,942.71","56,537,645.64","40,897,078,542.85","20,889,978,991.04","5,937,498,670.13",23.00,"771,857,059.33"
5498,IRL,2020,USA,"1,836,465.00","383,347,327,773.00","190,291,924,815.00","6,021,314,169.44","2,374,878,000,000.00","627,391,000,000.00","244,049,662,636.00",428.00,"19,796,405,431.00"
6134,KOR,2020,USA,"5,696,509.00","2,467,458,413,822.00","1,740,678,390,947.00","66,192,428,115.47","1,017,125,810,751.30","3,506,298,916,425.00","1,040,889,326,102.00","1,076.00","112,436,409,499.41"
6982,MAC,2020,USA,"29,892.00","2,324,792,198.00","13,239,593,333.00","620,734,467.32","3,287,923,263.00","2,483,678,112.00","158,885,914.00",0.00,"-370,183,201.00"
7406,MUS,2020,USA,"81,200.00","79,174,531,264.50","46,748,238,873.80","493,803.67","62,066,655,211.00","93,038,766,080.00","16,070,972,777.20",108.00,"-2,029,385,910.07"


### Step 4.5. Merge with the shares reported, and multiply the number

In [13]:
# Merge with share_reported_2020
excluded_jurisdictions_share_reported_2020 = pd.merge(excluded_jurisdictions_2020, shares_reported_2020, on='iso_partner', how='left')

# Multiply 'n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll",'stated_capital', 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip' and 'profit_loss_before_income_tax_corrected' by share_reported
excluded_jurisdictions_share_reported_2020['n_employees'] = excluded_jurisdictions_share_reported_2020['n_employees'] * excluded_jurisdictions_share_reported_2020['share_reported_total_n_employees_by_partner']
excluded_jurisdictions_share_reported_2020['unrelated_party_revenues'] = excluded_jurisdictions_share_reported_2020['unrelated_party_revenues'] * excluded_jurisdictions_share_reported_2020['share_reported_total_unrelated_party_revenues_by_partner']
excluded_jurisdictions_share_reported_2020['tangible_assets_except_cash'] = excluded_jurisdictions_share_reported_2020['tangible_assets_except_cash'] * excluded_jurisdictions_share_reported_2020['share_reported_total_tangible_assets_except_cash_by_partner']
excluded_jurisdictions_share_reported_2020['payroll'] = excluded_jurisdictions_share_reported_2020['payroll'] * excluded_jurisdictions_share_reported_2020['share_reported_total_payroll_by_partner']
excluded_jurisdictions_share_reported_2020['stated_capital'] = excluded_jurisdictions_share_reported_2020['stated_capital'] * excluded_jurisdictions_share_reported_2020['share_reported_total_stated_capital_by_partner']
excluded_jurisdictions_share_reported_2020['total_revenues'] = excluded_jurisdictions_share_reported_2020['total_revenues'] * excluded_jurisdictions_share_reported_2020['share_reported_total_total_revenues_by_partner']
excluded_jurisdictions_share_reported_2020['related_party_revenues'] = excluded_jurisdictions_share_reported_2020['related_party_revenues'] * excluded_jurisdictions_share_reported_2020['share_reported_total_related_party_revenues_by_partner']
excluded_jurisdictions_share_reported_2020['holding_or_managing_ip'] = excluded_jurisdictions_share_reported_2020['holding_or_managing_ip'] * excluded_jurisdictions_share_reported_2020['share_reported_total_holding_or_managing_ip_by_partner']
excluded_jurisdictions_share_reported_2020['profit_loss_before_income_tax_corrected'] = excluded_jurisdictions_share_reported_2020['profit_loss_before_income_tax_corrected'] * excluded_jurisdictions_share_reported_2020['share_reported_total_profit_loss_by_partner']

# Drop share_reported columns
excluded_jurisdictions_dataset_2020 = excluded_jurisdictions_share_reported_2020.drop(columns=[
    'share_reported_total_profit_loss_by_partner',
    'share_reported_total_n_employees_by_partner',
    'share_reported_total_unrelated_party_revenues_by_partner',
    'share_reported_total_tangible_assets_except_cash_by_partner',
    'share_reported_total_payroll_by_partner',
    'share_reported_total_stated_capital_by_partner',
    'share_reported_total_total_revenues_by_partner',
    'share_reported_total_related_party_revenues_by_partner',
    'share_reported_total_holding_or_managing_ip_by_partner'
])

excluded_jurisdictions_dataset_2020[excluded_jurisdictions_dataset_2020['iso_partner'] == 'USA']

,iso_parent,year,iso_partner,n_employees,unrelated_party_revenues,tangible_assets_except_cash,payroll,stated_capital,total_revenues,related_party_revenues,holding_or_managing_ip,profit_loss_before_income_tax_corrected
198,AUT,2020,USA,"429,393.98","177,632,609,189.37","83,535,543,651.76","8,846,994,707.19","57,833,932,941.50","210,117,993,370.12","36,266,703,604.60",50.54,"8,817,445,603.61"
410,CZE,2020,USA,"102,949.10","44,848,783,024.71","10,997,299,359.63","758,279,470.53","6,796,880,188.37","74,160,770,980.80","26,375,053,216.95",0.00,"1,515,546,336.22"
622,FIN,2020,USA,"205,902.28","126,724,787,886.74","35,759,862,915.68","2,918,651,521.29","138,855,982,087.90","161,181,508,269.05","35,074,273,891.94",17.51,"9,427,635,438.97"
834,GBR,2020,USA,"2,774,403.11","1,310,895,656,472.58","625,483,859,501.17","48,898,132,636.09","2,795,207,492,557.48","1,649,260,885,178.62","347,801,436,478.68",195.73,"92,748,029,273.45"
1046,HUN,2020,USA,"22,530.54","9,775,066,323.33","3,724,383,000.77","294,666,734.39","3,145,741,940.53","12,471,709,050.43","2,737,129,458.54",0.00,"487,628,443.38"
1258,IMN,2020,USA,"10,133.30","4,178,429,744.58","2,440,389,201.56","22,775,296.59","9,016,232,341.93","5,444,038,754.39","1,262,101,655.74",1.83,"196,548,048.42"
1470,IRL,2020,USA,"373,653.71","107,125,362,682.28","40,751,746,198.35","2,425,591,205.59","523,569,227,794.95","163,501,405,129.26","51,876,303,542.66",34.07,"5,041,017,383.00"
1682,KOR,2020,USA,"1,159,032.02","689,524,507,760.87","372,772,959,597.67","26,664,573,047.54","224,237,108,308.46","913,759,999,169.11","221,256,157,668.71",85.65,"28,631,152,091.95"
1894,MAC,2020,USA,"6,081.93","649,656,823.80","2,835,309,736.87","250,053,065.26","724,860,579.73","647,259,621.51","33,773,510.74",0.00,"-94,264,585.44"
2106,MUS,2020,USA,"16,521.24","22,125,106,300.29","10,011,314,813.58","198,921.00","13,683,309,517.64","24,246,393,374.42","3,416,118,887.49",8.60,"-516,769,051.09"


## Step 5. Calculate Misalignment with all countries reporting in the CBCR

### Step 5.1. Concatenate the two samples

- Concatenate the cbcr_sample (without the "bad reporters") and the sample with the bad reporters.
- Ensure all required columns are included: 'iso_parent', 'year', 'iso_partner', 'n_employees', 'unrelated_party_revenues', 'tangible_assets_except_cash', 'payroll', 'stated_capital', 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip', 'profit_loss_before_income_tax_corrected'.

In [14]:
final_misalignment_2020 = cbcr_sample[cbcr_sample['year'] == 2020].copy()

# Concatenate excluded_jurisdictions_dataset_2020
final_misalignment_2020 = pd.concat([final_misalignment_2020, excluded_jurisdictions_dataset_2020])

# Save the final misalignment dataset
#final_misalignment_2020.to_csv(f'{output_tables}/Datasets_to_Perform_Analysis/misalignment_dataset_2020.csv', index=False) 

final_misalignment_2020[final_misalignment_2020['iso_partner'] == 'ZAF']

,iso_parent,parent_jurisdiction,iso_partner,partner_jurisdiction,year,unrelated_party_revenues,profit_loss_before_income_tax,adjusted_profit_loss_before_income_tax,income_tax_paid_on_cash_basis,income_tax_accrued_current_year,n_employees,tangible_assets_except_cash,stated_capital,total_revenues,related_party_revenues,holding_or_managing_ip,n_cbcr,n_cbcr_groups,n_entities,profit_loss_before_income_tax_corrected,ln_profit_loss_before_income_tax_corrected,ln_unrelated_party_revenues,ln_n_employees,ln_tangible_assets_except_cash,ln_stated_capital,ln_total_revenues,ln_related_party_revenues,ln_holding_or_managing_ip,etr_domestic,etr_domestic_corrected,etr_foreign,etr_foreign_corrected,etr_average,etr_average_corrected,cit,gdp_current_usd,population,gdp,wage_monthly,payroll,ln_wage_monthly,ln_gdp_current_usd,ln_population,gvt_health_expenditure,ln_gvt_health_expenditure,tax_revenue_pct_gdp,tax_revenue_current_usd,cthi_2021_share,cthi_2021_score,region_tjn,ukt,gbr_oct,nld_oct,oecd_oct,oecd,eu
750,AUS,Australia,ZAF,South Africa,2020,"2,506,649,792.00","27,750,506.37",NaN,"69,779,812.13","82,003,827.20","15,670.00","1,999,994,967.00","4,653,892,533.00","3,771,076,461.00","1,264,426,669.00",4.00,29.00,29.00,156.00,"27,750,506.37",17.14,21.64,9.66,21.42,22.26,22.05,20.96,1.61,0.11,0.13,0.19,0.21,0.12,0.14,0.28,"338,291,396,026.70","58,801,927.00",NaN,484.85,"91,171,758.12",6.19,26.55,17.89,"17,882,857,924.61",23.61,23.24,"78,629,614,436.56",0.00,49.42,Africa,0.00,0.00,0.00,0.00,0.00,0.00
1005,BEL,Belgium,ZAF,South Africa,2020,"1,822,000,000.00","-169,400,000.00",NaN,"31,300,000.00","26,100,000.00","9,900.00","1,728,100,000.00","131,100,000.00","2,414,600,000.00","592,600,000.00",0.00,19.00,19.00,64.00,"-169,400,000.00",0.00,21.32,9.20,21.27,18.69,21.60,20.20,0.00,0.11,0.13,0.19,0.21,0.12,0.14,0.28,"338,291,396,026.70","58,801,927.00",NaN,484.85,"57,600,536.40",6.19,26.55,17.89,"17,882,857,924.61",23.61,23.24,"78,629,614,436.56",0.00,49.42,Africa,0.00,0.00,0.00,0.00,0.00,0.00
1642,BMU,Bermuda,ZAF,South Africa,2020,"190,695,193.50","7,802,276.05",NaN,"709,361.05","1,152,215.79",933.00,"42,298,362.53","57,823,577.64","300,521,213.20","109,826,018.70",NaN,15.00,15.00,63.00,"7,802,276.05",15.87,19.07,6.84,17.56,17.87,19.52,18.51,NaN,0.11,0.13,0.19,0.21,0.12,0.14,0.28,"338,291,396,026.70","58,801,927.00",NaN,484.85,"5,428,414.19",6.19,26.55,17.89,"17,882,857,924.61",23.61,23.24,"78,629,614,436.56",0.00,49.42,Africa,0.00,0.00,0.00,0.00,0.00,0.00
1891,BRA,Brazil,ZAF,South Africa,2020,"464,525,000.00","26,843,000.00",NaN,"11,055,000.00","11,926,000.00","2,154.00","198,691,000.00","54,960,090,000.00","557,188,000.00","92,663,000.00",0.00,12.00,12.00,35.00,"26,843,000.00",17.11,19.96,7.68,19.11,24.73,20.14,18.34,0.00,0.11,0.13,0.19,0.21,0.12,0.14,0.28,"338,291,396,026.70","58,801,927.00",NaN,484.85,"12,532,480.34",6.19,26.55,17.89,"17,882,857,924.61",23.61,23.24,"78,629,614,436.56",0.00,49.42,Africa,0.00,0.00,0.00,0.00,0.00,0.00
2712,CHE,Switzerland,ZAF,South Africa,2020,"9,176,485,514.00","-1,112,928,013.00",NaN,"107,848,523.20","-4,414,644.83","33,392.00","6,761,492,980.00","1,139,757,160.00","11,015,455,933.00","1,838,969,418.00",1.00,59.00,59.00,269.00,"-1,112,928,013.00",0.00,22.94,10.42,22.63,20.85,23.12,21.33,0.69,0.11,0.13,0.19,0.21,0.12,0.14,0.28,"338,291,396,026.70","58,801,927.00",NaN,484.85,"194,282,536.51",6.19,26.55,17.89,"17,882,857,924.61",23.61,23.24,"78,629,614,436.56",0.00,49.42,Africa,0.00,0.00,0.00,0.00,0.00,0.00
3562,CHN,China (People’s Republic of),ZAF,South Africa,2020,"4,133,801,676.00","401,660,907.50",NaN,"91,994,288.60","157,461,677.70","9,795.00","2,910,917,721.00","1,253,366,007.00","5,780,543,034.00","1,646,741,357.00",8.00,87.00,87.00,155.00,"401,660,907.50",19.81,22.14,9.19,21.79,20.95,22.48,21.22,2.20,0.11,0.13,0.19,0.21,0.12,0.14,0.28,"338,291,396,026.70","58,801,927.00",NaN,484.85,"56,989,621.62",6.19,26.55,17.89,"17,882,857,924.61",23.61,23.24,"78,629,614,436.56",0.00,49.42,Africa,0.00,0.00,0.00,0.00,0.00,0.00
4

### Step 5.2. Perform the misalignment estimates, and all the remaining calculations needed

In [15]:
# Initialize a list to store the aggregate results
results_sample = []

# Start the estimates
misalignment_final_estimates_2020 = final_misalignment_2020[final_misalignment_2020['year'] == 2020].copy()
misalignment_final_estimates_2020 = calculate_misalignment(misalignment_final_estimates_2020, etr_max=0.15, weights=[1/2, 0, 0, 1/2, 0, 0, 0, 0])

# Perform the groupby operation on 'iso_partner'
country_results_2020 = misalignment_final_estimates_2020.groupby(['iso_partner']).agg(
    negative_misalignment=('misaligned_profit', lambda x: x[x < 0].sum()),
    positive_misalignment=('misaligned_profit', lambda x: x[x > 0].sum()),
    theoretical_profit=('theoretical_profit', 'sum'),
    reported_profit=('profit_loss_before_income_tax_corrected', 'sum')
).reset_index()

# Convert results to millions
country_results_2020['negative_misalignment'] = -country_results_2020['negative_misalignment'] / 1e6
country_results_2020['positive_misalignment'] = country_results_2020['positive_misalignment'] / 1e6
country_results_2020['theoretical_profit'] = country_results_2020['theoretical_profit'] / 1e6
country_results_2020['reported_profit'] = country_results_2020['reported_profit'] / 1e6

# Merge the unique columns back into the result
country_results_2020 = country_results_2020.merge(unique_columns, on='iso_partner', how='left')

# Calculate other relevant variables
country_results_2020['tax_revenue_loss'] = country_results_2020['negative_misalignment'] * country_results_2020['cit']
country_results_2020['tax_revenue_gain'] = country_results_2020['positive_misalignment'] * country_results_2020['etr_average_corrected']

country_results_2020['tax_revenue_loss_pct_of_gvt_health_expenditure'] = np.where(
    country_results_2020['gvt_health_expenditure'] == 0, 
    np.nan, 
    country_results_2020['tax_revenue_loss'] / (country_results_2020['gvt_health_expenditure'] / 1e6)
)
    
country_results_2020['tax_revenue_loss_pct_of_total_tax_revenues'] = np.where(
    country_results_2020['tax_revenue_current_usd'] == 0, 
    np.nan, 
    country_results_2020['tax_revenue_loss'] / (country_results_2020['tax_revenue_current_usd'] / 1e6)
)

# Calculate totals
total_positive_misalignment = country_results_2020['positive_misalignment'].sum()
total_negative_misalignment = country_results_2020['negative_misalignment'].sum()
total_profits = country_results_2020['reported_profit'].sum()
misaligned_of_total_profits = total_positive_misalignment / total_profits
total_tax_revenue_loss = country_results_2020['tax_revenue_loss'].sum()
total_tax_revenue_gain = country_results_2020['tax_revenue_gain'].sum()
average_tax_revenue_loss_pct_of_gvt_health_expenditure = country_results_2020['tax_revenue_loss_pct_of_gvt_health_expenditure'].mean()
average_tax_revenue_loss_pct_of_total_tax_revenues = country_results_2020['tax_revenue_loss_pct_of_total_tax_revenues'].mean()

print(f"Year {2020}: Positive Misalignment: {total_positive_misalignment}, Negative Misalignment: {total_negative_misalignment}, Shifted of total profits: {misaligned_of_total_profits}, "
        f"Total tax revenue loss: {total_tax_revenue_loss}, Total tax revenue gain: {total_tax_revenue_gain}")

# Calculate countries' fractions of totals
country_results_2020['tax_revenue_loss_caused_pct_of_total'] = country_results_2020['positive_misalignment'] / total_positive_misalignment
country_results_2020['tax_revenue_loss_caused_usd'] = country_results_2020['tax_revenue_loss_caused_pct_of_total'] * total_tax_revenue_loss
country_results_2020['tax_revenue_loss_suffered_pct_of_total'] = country_results_2020['tax_revenue_loss'] / total_tax_revenue_loss

country_results_2020 = country_results_2020.sort_values(by='iso_partner')
country_results_2020.to_csv(f'{output_tables}/Final_Full_CBCR_Datasets/SOTJ_sample_countries_2020.csv', index=False) # CHANGE FILE NAME HERE, DEPENDING ON FORMULA USE
    
# Append aggregate results to the list
results_sample.append({
    'year': 2020,
    'total_positive_misalignment': total_positive_misalignment,
    'total_negative_misalignment': total_negative_misalignment,
    'total_profits': total_profits,
    'misaligned_of_total_profits': misaligned_of_total_profits,
    'total_tax_revenue_loss': total_tax_revenue_loss,
    'total_tax_revenue_gain': total_tax_revenue_gain,
    'average_tax_revenue_loss_pct_of_gvt_health_expenditure': average_tax_revenue_loss_pct_of_gvt_health_expenditure,
    'average_tax_revenue_loss_pct_of_total_tax_revenues': average_tax_revenue_loss_pct_of_total_tax_revenues
})

# Convert aggregate results to a DataFrame
results_sample_df = pd.DataFrame(results_sample)

# Save the aggregated results to a CSV or Excel file
results_sample_df.to_csv(f'{output_tables}/Final_Full_CBCR_Datasets/SOTJ_sample_aggregate_results_2020.csv', index=False)  # CHANGE FILE NAME HERE, DEPENDING ON FORMULA USE


Year 2020: Positive Misalignment: 1059917.5949892718, Negative Misalignment: 1059917.594989272, Shifted of total profits: 0.23924047479579463, Total tax revenue loss: 266431.73961564805, Total tax revenue gain: 91588.62835497994


/var/folders/pm/bp4z4lln39xcwn73chrtp3n00000gn/T/ipykernel_6472/421381151.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent', group_keys=False).apply(lambda x: adjust_misalignment(x)).reset_index(drop=True)


## Step 6. Checking the datasets

### Step 6.1. Checking the countries

In [16]:
# Open the CSV file f'{output_tables}/Scaling_Mario/SOTJ_sample_2016.csv'
sotj_2020_countries = pd.read_csv(f'{output_tables}/Final_Full_CBCR_Datasets/SOTJ_sample_countries_2020.csv')
sotj_2020_countries

,iso_partner,negative_misalignment,positive_misalignment,theoretical_profit,reported_profit,partner_jurisdiction,etr_average_corrected,cit,tax_revenue_current_usd,gvt_health_expenditure,region_tjn,ukt,oecd,oecd_oct,nld_oct,tax_revenue_loss,tax_revenue_gain,tax_revenue_loss_pct_of_gvt_health_expenditure,tax_revenue_loss_pct_of_total_tax_revenues,tax_revenue_loss_caused_pct_of_total,tax_revenue_loss_caused_usd,tax_revenue_loss_suffered_pct_of_total
0,ABW,51.34,0.03,39.21,-14.51,Aruba,0.25,0.25,NaN,NaN,Caribbean/American isl.,0.00,0.00,1.00,1.00,12.83,0.01,NaN,NaN,0.00,0.01,0.00
1,AFG,19.19,0.85,13.51,-8.77,Afghanistan,0.07,0.20,NaN,"238,944,014.08",Asia,0.00,0.00,0.00,0.00,3.84,0.06,0.02,NaN,0.00,0.21,0.00
2,AGO,"1,424.21",0.91,354.82,"-1,216.07",Angola,0.35,0.30,NaN,"971,570,187.31",Africa,0.00,0.00,0.00,0.00,427.26,0.32,0.44,NaN,0.00,0.23,0.00
3,AIA,0.64,23.07,-23.07,-1.00,Anguilla,0.00,0.00,NaN,NaN,Caribbean/American isl.,1.00,0.00,1.00,0.00,0.00,0.00,NaN,NaN,0.00,5.80,0.00
4,ALB,2.75,29.90,20.57,46.29,Albania,0.09,0.15,"2,575,126,921.93","453,366,123.39",Europe,0.00,0.00,0.00,0.00,0.41,2.83,0.00,0.00,0.00,7.51,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
207,XKV,12.89,0.01,24.54,2.58,Kosovo,0.08,0.10,NaN,NaN,NaN,0.00,0.00,0.00,0.00,1.29,0.00,NaN,NaN,0.00,0.00,0.00
208,YEM,131.12,3.07,166.63,20.88,Yemen,0.45,0.20,NaN,NaN,Asia,0.00,0.00,0.00,0.00,26.22,1.37,NaN,NaN,0.00,0.77,0.00
209,ZAF,"4,030.93","10,197.09","8,061.04","12,741.89",South Africa,0.14,0.28,"78,629,614,436.56","17,882,857,924.61",Africa,0.00,0.00,0.00,0.00,"1,128.66","1,452.93",0.06,0.01,0.01,"2,563.25",0.00
210,ZMB,"1,520.62",29.83,582.58,"-1,299.81",Zambia,0.14,0.35,"2,977,859,245.81","651,808,362.64",Africa,0.00,0.00,0.00,0.00,532.22,4.31,0.82,0.18,0.00,7.50,0.00


### Step 6.2. Checking the aggregate results

In [17]:
sotj_2020_aggregate = pd.read_csv(f'{output_tables}/Final_Full_CBCR_Datasets/SOTJ_sample_aggregate_results_2020.csv')
sotj_2020_aggregate


,year,total_positive_misalignment,total_negative_misalignment,total_profits,misaligned_of_total_profits,total_tax_revenue_loss,total_tax_revenue_gain,average_tax_revenue_loss_pct_of_gvt_health_expenditure,average_tax_revenue_loss_pct_of_total_tax_revenues
0,2020,"1,059,917.59","1,059,917.59","4,430,343.97",0.24,"266,431.74","91,588.63",0.34,0.04
